# 02 -- JWT et OAuth2

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre la structure d'un JWT (header, payload, signature) ;
- encoder et decoder un JWT avec `PyJWT` ;
- gerer l'expiration, l'issuer, l'audience ;
- choisir entre HS256 (symetrique) et RS256 (asymetrique) ;
- comprendre les flows OAuth2 (Authorization Code, Client Credentials) ;
- proteger une API FastAPI avec des JWT ;
- eviter les erreurs classiques (alg=none, refresh token mal gere).

## Prerequis -- ce que vous devez maitriser

- fonctions typees, decorateurs ;
- HTTP (headers, codes de statut) ;
- JSON ;
- bases de la cryptographie (hash, signature).

Ce que ce notebook **ne couvrira pas** :

- l'implementation d'un serveur OAuth2 complet ;
- les sessions cookie (voir `02_Flask_complet.ipynb`) ;
- SAML et OpenID Connect en detail.

## Plan

1. Structure d'un JWT
2. Installation PyJWT
3. Encoder un JWT
4. Decoder et verifier
5. Claims standards
6. Expiration et nbf
7. HS256 vs RS256
8. Cles asymetriques
9. Vulnerabilites a eviter
10. OAuth2 : concepts
11. Authorization Code Flow
12. Client Credentials Flow
13. Proteger une API FastAPI
14. Refresh tokens
15. Synthese
16. Exercices

---

## 1. Structure d'un JWT

Un JWT est compose de **3 parties** separees par des points :

```
<header>.<payload>.<signature>
```

Chaque partie est encodee en **base64url**. Exemple :

```
eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.
eyJzdWIiOiIxMjM0In0.
SflKxwRJSMeKKF2QT4fwpMeJf36POk6yJV_adQssw5c
```

- **header** : algo et type (`{"alg": "HS256", "typ": "JWT"}`)
- **payload** : claims (donnees) (`{"sub": "1234"}`)
- **signature** : HMAC ou signature asymetrique

---

## 2. Installation PyJWT

In [ ]:
# pip install pyjwt cryptography
import jwt
jwt.__version__

---

## 3. Encoder un JWT

In [ ]:
payload = {'user_id': 42, 'role': 'admin'}
token = jwt.encode(payload, 'secret', algorithm='HS256')
token

---

## 4. Decoder et verifier

In [ ]:
decoded = jwt.decode(token, 'secret', algorithms=['HS256'])
decoded

In [ ]:
# Mauvaise cle : erreur
try:
    jwt.decode(token, 'wrong', algorithms=['HS256'])
except jwt.InvalidSignatureError as e:
    print('Erreur :', e)

---

## 5. Claims standards

| Claim | Nom | Role |
|---|---|---|
| `iss` | issuer | emetteur (ex. `mon-api`) |
| `sub` | subject | sujet (ex. user id) |
| `aud` | audience | destinataire |
| `exp` | expiration time | timestamp UNIX |
| `nbf` | not before | valide apres ce timestamp |
| `iat` | issued at | date de creation |
| `jti` | JWT ID | identifiant unique |

In [ ]:
import datetime
now = datetime.datetime.now(datetime.UTC)
payload = {
    'sub': '1234',
    'iss': 'mon-api',
    'aud': 'mobile-app',
    'iat': now,
    'exp': now + datetime.timedelta(hours=1),
}
token = jwt.encode(payload, 'secret', algorithm='HS256')
jwt.decode(token, 'secret', algorithms=['HS256'], audience='mobile-app')

---

## 6. Expiration et nbf

In [ ]:
import datetime
expired = jwt.encode(
    {'sub': '1', 'exp': datetime.datetime.now(datetime.UTC) - datetime.timedelta(seconds=1)},
    'secret', algorithm='HS256'
)
try:
    jwt.decode(expired, 'secret', algorithms=['HS256'])
except jwt.ExpiredSignatureError:
    print('Token expire')

---

## 7. HS256 vs RS256

### HS256 (HMAC-SHA256) - Symetrique

- Une seule cle (secret partage)
- Rapide
- Celui qui verifie peut aussi signer
- OK si emetteur = verificateur

### RS256 (RSA-SHA256) - Asymetrique

- Paire cle privee / cle publique
- Le serveur signe avec la privee, les clients verifient avec la publique
- Indispensable pour services distribues / SSO
- Utilise par Google, Auth0, Keycloak

---

## 8. Cles asymetriques

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

private = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public = private.public_key()

pem_priv = private.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
pem_pub = public.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)
print('Cles generees')

In [ ]:
token_rs256 = jwt.encode({'sub': 'alice'}, pem_priv, algorithm='RS256')
jwt.decode(token_rs256, pem_pub, algorithms=['RS256'])

---

## 9. Vulnerabilites a eviter

### CVE-2015-9235 : `alg=none`

Un attaquant modifie le header pour mettre `{"alg": "none"}` et supprime la signature. Certaines lib acceptaient !

**Regle** : TOUJOURS specifier la liste `algorithms=['HS256']` au decode, jamais accepter `none`.

### Confusion HS256/RS256

Si vous acceptez `algorithms=['HS256', 'RS256']`, un attaquant peut prendre votre **cle publique** et signer un token en HS256 (HMAC avec la cle publique).

**Regle** : un seul algo dans `algorithms=`.

### Secret faible

Pas de `secret`, `password`, `123` en production. Generez avec `secrets.token_urlsafe(32)`.

In [ ]:
import secrets
secrets.token_urlsafe(32)

---

## 10. OAuth2 : concepts

**OAuth2** est un protocole de delegation d'authorization. Acteurs :

- **Resource Owner** : l'utilisateur
- **Client** : l'application qui veut acceder
- **Authorization Server** : emet les tokens
- **Resource Server** : expose l'API protegee

**Flows principaux** :

| Flow | Usage |
|---|---|
| Authorization Code | Web app classique (avec backend) |
| Authorization Code + PKCE | SPA, mobile |
| Client Credentials | Service-a-service (pas d'utilisateur) |
| Device Code | TV, CLI sans navigateur |
| Password (deprecated) | Deprecie (trop permissif) |

---

## 11. Authorization Code Flow

```
1. [User] clique 'Login with Google'
2. [Client] redirige vers /authorize?client_id=...&redirect_uri=...&scope=...
3. [AuthServer] affiche login + consentement
4. [AuthServer] redirige vers redirect_uri?code=ABC
5. [Client] echange code contre token : POST /token
   { code, client_id, client_secret, redirect_uri }
6. [AuthServer] retourne { access_token, refresh_token, id_token }
7. [Client] appelle l'API : Authorization: Bearer <access_token>
```

---

## 12. Client Credentials Flow

In [ ]:
# Pseudo-code d'un service qui veut appeler une API
# POST https://auth.example.com/token
# grant_type=client_credentials&client_id=X&client_secret=Y&scope=read:users
# -> { 'access_token': '...', 'token_type': 'Bearer', 'expires_in': 3600 }
print('Ce flow ne requiert pas d\'utilisateur : uniquement client_id + client_secret')

---

## 13. Proteger une API FastAPI

In [ ]:
from fastapi import FastAPI, HTTPException, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
import jwt

api = FastAPI()
security = HTTPBearer()
SECRET = 'change-me'

def get_current_user(creds: HTTPAuthorizationCredentials = Depends(security)) -> dict:
    try:
        return jwt.decode(creds.credentials, SECRET, algorithms=['HS256'])
    except jwt.ExpiredSignatureError:
        raise HTTPException(401, 'token expire')
    except jwt.InvalidTokenError:
        raise HTTPException(401, 'token invalide')

@api.get('/me')
def me(user: dict = Depends(get_current_user)) -> dict:
    return {'user_id': user['sub']}

print('API protegee')

---

## 14. Refresh tokens

In [ ]:
# Usage typique :
# - access_token court (15 min)
# - refresh_token long (7 jours), stocke en DB ou cookie HttpOnly
# - endpoint /refresh qui echange refresh contre un nouveau access_token

import datetime
now = datetime.datetime.now(datetime.UTC)
access = jwt.encode({'sub': '1', 'type': 'access', 'exp': now + datetime.timedelta(minutes=15)}, 'secret', algorithm='HS256')
refresh = jwt.encode({'sub': '1', 'type': 'refresh', 'exp': now + datetime.timedelta(days=7)}, 'secret', algorithm='HS256')
print('access :', access[:30], '...')
print('refresh:', refresh[:30], '...')

---

## Synthese

| Operation | PyJWT |
|---|---|
| Encoder | `jwt.encode(payload, secret, algorithm='HS256')` |
| Decoder | `jwt.decode(token, secret, algorithms=['HS256'])` |
| Expiration | `exp` claim = datetime UTC |
| Audience | `aud=...` au decode |
| RSA | `algorithm='RS256'` avec cle privee/publique |
| Erreurs | `ExpiredSignatureError`, `InvalidTokenError` |

### Regles a retenir

- **Toujours specifier `algorithms=`** au decode (jamais `alg=none`).
- **Un seul algo** a la fois (pas de mix HS/RS).
- **Secret fort** : `secrets.token_urlsafe(32)`.
- **Access token court** + refresh token long.
- **OAuth2 = flows standardises** : Auth Code + PKCE pour SPA/mobile.
- **Ne jamais stocker un token en localStorage** (XSS).

---

## Exercices

### Exercice 1 (facile) -- Token simple

Creez un token pour `user_id=123, role='user'` avec secret `mySecret` et decodez-le.

In [ ]:
import jwt

# TODO

<details>
<summary>Voir la correction</summary>

```python
import jwt

payload = {'user_id': 123, 'role': 'user'}
token = jwt.encode(payload, 'mySecret', algorithm='HS256')
decoded = jwt.decode(token, 'mySecret', algorithms=['HS256'])
print(decoded)
```

</details>

### Exercice 2 (facile) -- Expiration a 5 minutes

Generez un token qui expire dans 5 minutes. Verifiez qu'il est encore valide juste apres emission.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import jwt
import datetime

now = datetime.datetime.now(datetime.UTC)
token = jwt.encode(
    {'sub': '1', 'exp': now + datetime.timedelta(minutes=5)},
    'secret', algorithm='HS256',
)
print(jwt.decode(token, 'secret', algorithms=['HS256']))
```

</details>

### Exercice 3 (moyen) -- Fonction create_token / verify_token

Ecrivez deux fonctions typees :

- `create_token(user_id: int, ttl_minutes: int) -> str`
- `verify_token(token: str) -> dict`

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import jwt
import datetime

SECRET = 'change-me'

def create_token(user_id: int, ttl_minutes: int) -> str:
    now = datetime.datetime.now(datetime.UTC)
    payload = {'sub': str(user_id), 'iat': now, 'exp': now + datetime.timedelta(minutes=ttl_minutes)}
    return jwt.encode(payload, SECRET, algorithm='HS256')

def verify_token(token: str) -> dict:
    return jwt.decode(token, SECRET, algorithms=['HS256'])

t = create_token(42, 10)
print(verify_token(t))
```

</details>

### Exercice 4 (moyen) -- RSA

Generez une paire de cles RSA 2048, signez un payload, decodez avec la cle publique.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import jwt
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

priv = rsa.generate_private_key(public_exponent=65537, key_size=2048)
pub = priv.public_key()
priv_pem = priv.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
pub_pem = pub.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)

token = jwt.encode({'sub': 'alice'}, priv_pem, algorithm='RS256')
print(jwt.decode(token, pub_pem, algorithms=['RS256']))
```

</details>

### Exercice 5 (difficile) -- Depends FastAPI avec roles

Creez un decorateur/dependance FastAPI `require_role(role)` qui rejette en 403 si le JWT ne contient pas le bon role.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from fastapi import FastAPI, HTTPException, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
import jwt

api = FastAPI()
security = HTTPBearer()
SECRET = 'change-me'

def require_role(role: str):
    def dep(creds: HTTPAuthorizationCredentials = Depends(security)) -> dict:
        try:
            payload = jwt.decode(creds.credentials, SECRET, algorithms=['HS256'])
        except jwt.InvalidTokenError:
            raise HTTPException(401, 'invalid token')
        if payload.get('role') != role:
            raise HTTPException(403, 'forbidden')
        return payload
    return dep

@api.get('/admin')
def admin_only(user: dict = Depends(require_role('admin'))) -> dict:
    return {'welcome': user['sub']}
```

</details>

---

## Ressources externes

- [PyJWT documentation](https://pyjwt.readthedocs.io/)
- [jwt.io - debugger + bibliotheques](https://jwt.io/)
- [RFC 7519 - JSON Web Token](https://datatracker.ietf.org/doc/html/rfc7519)
- [OAuth 2.0](https://oauth.net/2/)